# RGIS label detector -- Colab training (small / medium / large)

No local GPU, so this trains on Colab instead. Mirrors `training/prepare_dataset.py`,
`train.py` and `export.py` in the repo -- same hyperparameters and reasoning, which is
documented in `training/README.md`.

**Before running:**
1. `Runtime` -> `Change runtime type` -> GPU (T4 is fine).
2. Locally, zip your `dataset2/` folder (`images/`, `labels/`, `classes.txt`,
   `notes.json`) into `dataset2.zip`.
3. Run the cells top to bottom. Sections 4 and 5 (YOLO11n / YOLO26n training) are
   separate on purpose -- run one, look at its results, decide whether the second is
   worth running, rather than both auto-chaining.


## 0. Confirm the GPU runtime

In [1]:
!nvidia-smi


Tue Aug 11 04:08:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 592.82                 Driver Version: 592.82         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   55C    P0             13W /   70W |       0MiB /   6144MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Install dependencies

In [2]:
!pip install -q -U ultralytics pyyaml onnxruntime

import ultralytics
ultralytics.checks()


Ultralytics 8.4.117  Python-3.14.0 torch-2.13.0+cpu CPU (13th Gen Intel Core i5-13420H)
Setup complete  (12 CPUs, 31.6 GB RAM, 445.9/475.7 GB disk)


## 4. Train YOLO11n

Run this section, wait for it to finish, then check the results below before
deciding whether to also run YOLO26n in section 5.

In [ ]:
!yolo task=detect mode=train data="dataset/data.yaml" model=yolo11n.pt epochs=250 imgsz=960

In [ ]:
from ultralytics import YOLO

# Same defaults as training/train.py -- see training/README.md for the reasoning
# behind scale/degrees specifically (default scale augmentation fights a
# size-based class scheme; the tags are photographed at an angle, hence degrees>0).
COMMON_TRAIN_ARGS = dict(
    data=str('dataset/data.yaml'),
    epochs=250,
    imgsz=960,        # matches AppConstants.modelInputSize in the Flutter app
    batch=16,
    patience=50,
    seed=42,
    pretrained=True,
    scale=0.3,
    degrees=10.0,
    project='/content/runs',
)

model_11n = YOLO('yolo11n.pt')
results_11n = model_11n.train(name='yolo11n_detect', **COMMON_TRAIN_ARGS)

In [ ]:
model_26n = YOLO('yolo26n.pt')
results_26n = model_26n.train(name='yolo26n_detect', **COMMON_TRAIN_ARGS)

### results

In [4]:
!yolo task=detect mode=val model="../yolo/content/runs/detect/train/weights/best.pt" data=../yolo/dataset2_training/data.yaml

Ultralytics 8.4.117  Python-3.14.0 torch-2.13.0+cpu CPU (13th Gen Intel Core i5-13420H)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 850.8303.2 MB/s, size: 5434.9 KB)

val: Scanning C:\Users\hassa\Downloads\Data\Hassan\Learning\ML Courses\Projects\RGIS_Flutter_IOS\yolo\dataset2_training\valid\labels... 5 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5/5 375.9it/s 0.0s
val: New cache created: C:\Users\hassa\Downloads\Data\Hassan\Learning\ML Courses\Projects\RGIS_Flutter_IOS\yolo\dataset2_training\valid\labels.cache

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.7s/it 1.7s
                   all          5        154      0.763      0.817        0.8       0.66
                 large          2          2      0.665      0.994      0.828      0.762
                medium          5         45      0.796      0.607      0.739      0

In [5]:
!yolo task=detect mode=val model="../yolo/content/runs/yolo11n_detect/weights/best.pt" data=../yolo/dataset2_training/data.yaml

Ultralytics 8.4.117  Python-3.14.0 torch-2.13.0+cpu CPU (13th Gen Intel Core i5-13420H)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1474.5167.0 MB/s, size: 5434.9 KB)

val: Scanning C:\Users\hassa\Downloads\Data\Hassan\Learning\ML Courses\Projects\RGIS_Flutter_IOS\yolo\dataset2_training\valid\labels.cache... 5 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5/5 911.8Kit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.6s/it 1.6s
                   all          5        154      0.845      0.767      0.843      0.681
                 large          2          2      0.839          1      0.995      0.846
                medium          5         45      0.842      0.591       0.73      0.572
                 small          4        107      0.854      0.709      0.804      0.625
Speed: 3.2ms preprocess, 151.2ms inference, 0.0

In [7]:
!yolo task=detect mode=val model="../yolo/content/runs/yolo26n_detect/weights/best.pt" data=../yolo/dataset2_training/data.yaml

Ultralytics 8.4.117  Python-3.14.0 torch-2.13.0+cpu CPU (13th Gen Intel Core i5-13420H)
YOLO26n summary (fused): 122 layers, 2,375,421 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1717.8150.4 MB/s, size: 5434.9 KB)

val: Scanning C:\Users\hassa\Downloads\Data\Hassan\Learning\ML Courses\Projects\RGIS_Flutter_IOS\yolo\dataset2_training\valid\labels.cache... 5 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5/5 873.8Kit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5s/it 1.5s
                   all          5        154      0.593      0.791      0.666      0.561
                 large          2          2      0.372      0.906      0.448      0.404
                medium          5         45       0.66      0.644      0.743      0.597
                 small          4        107      0.746      0.821      0.806      0.683
Speed: 2.3ms preprocess, 139.5ms inference, 0.0

In [ ]:
from IPython.display import Image, display

display(Image(filename='/content/runs/detect/train/results.png'))
metrics_11n = model_11n.val()
print('per-class mAP50-95:', dict(zip(metrics_11n.names.values(), metrics_11n.box.maps)))


In [ ]:
display(Image(filename='/content/runs/yolo26n_detect/results.png'))
metrics_26n = model_26n.val()
print('per-class mAP50-95:', dict(zip(metrics_26n.names.values(), metrics_26n.box.maps)))


## 6.Export to ONNX

Set `BEST_WEIGHTS` to whichever run generalized better (compare the `results.png`
plots and per-class mAP printed above -- with only ~33 source images, don't be
surprised if the difference is mostly noise). Mirrors `training/export.py`.

In [5]:
from ultralytics import YOLO

BEST_WEIGHTS = '../yolo/yolo11n-best.pt'  # change to yolo26n_detect if it won

export_model = YOLO(BEST_WEIGHTS)
assert export_model.task == 'detect', f"task is '{export_model.task}' -- the app only decodes a plain detect head"

exported_path = export_model.export(format='onnx', imgsz=960, opset=12, nms=False, simplify=True)
# exported_path = export_model.export(format='onnx', imgsz=960, opset=12, nms=False, simplify=True,
#                   int8=True, data='dataset/data.yaml')  # needs val images to calibrate

print(f'exported: {exported_path}')

import onnxruntime as ort

session = ort.InferenceSession(str(exported_path), providers=['CPUExecutionProvider'])
for out in session.get_outputs():
    print(f"output '{out.name}': shape={out.shape}")
    
print("Expected by onnx_detection_engine.dart: [1, 7, N] for 3 classes (4 box coords "
      "+ 3 class scores). If this doesn't match -- most likely with YOLO26's "
      "end-to-end/NMS-free head -- don't wire it into the app yet; see the YOLO26 "
      "note in training/README.md.")


Ultralytics 8.4.117  Python-3.14.0 torch-2.13.0+cpu CPU (13th Gen Intel Core i5-13420H)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 14.5 GFLOPs

PyTorch: starting from '..\yolo\yolo11n-best.pt' with input shape (1, 3, 960, 960) BCHW and output shape(s) (1, 7, 18900) (5.3 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/17.2 MB ? eta -:--:--
    --------------------------------------- 0.3/17.2 MB ? eta -:--:--
   - -------------------------------------- 0.8/17.2 MB 2.4 MB/s eta 0:00:07
   --- ------------------------------------ 1.6/17.2 MB 3.4 MB/s eta 0:00:05
   ----- ---------------------------------- 2.4/17.2 MB 3.3 MB/s eta 0:00:05
   ------- -------------------------------- 3.4/17.2 MB 3.4 MB/s e